In [32]:
from dotenv import load_dotenv
import os
from typing import List, Dict, Any, Optional, Union

import pandas as pd
import numpy as np
from pathlib import Path
from tqdm.notebook import tqdm

load_dotenv()


# Import from our Classes module
from Classes.model_classes import SQLLineageExtractor
from Classes.helper_classes import SQLLineageResult
from Classes.regexp_extractor import RegexSQLExtractor
from Classes.validation_classes import SQLLineageValidator




# Get the current working directory
current_dir = Path.cwd()
# Construct the path relative to current directory
file_path = current_dir / 'data' / 'views.csv'
wrong_sql_path = current_dir / 'data' / 'worng_sql_parsed.csv'

GIGACHAT_CREDENTIALS = os.environ.get("GIGACHAT_CREDENTIALS")

In [12]:
#Create a validaition Class
validation = SQLLineageValidator()

# Create Regexp extractor
re_extractor = RegexSQLExtractor()

extractor = SQLLineageExtractor(
    credentials=GIGACHAT_CREDENTIALS,   # or set env var GIGACHAT_API_KEY
    model="GigaChat",
    verify_ssl_certs=False,
    temperature=0.01
)

In [13]:
df_data = pd.read_csv(file_path)
df_data['ddl'] = "INSERT INTO s_grnplm_vd_t_bvd_db_dmslcl." + df_data['table_name'] + " " + df_data['view_def']

In [14]:
data_lineage = []
f1_scores = {}
results = []

for index, row in tqdm(df_data.iterrows(), total=len(df_data), desc="🎨 Extracting S2T"):

    target_ = re_extractor.extract(row['ddl'])

    res_ = await validation.run_comprehensive_validation(
        extractor, 
        row['ddl'],
        expected_result=target_)
    results.append(res_)
    try:
        f1_scores[row['table_name']] = res_['metrics']['f1_score']
    except:
        f1_scores[row['table_name']] = 0
        
    data_lineage.append(res_['result'])

🎨 Extracting S2T:   0%|          | 0/127 [00:00<?, ?it/s]

Giga generation stopped with reason: length


In [15]:
np.mean(list(f1_scores.values()))

np.float64(0.7897890754086361)

In [16]:
df = pd.DataFrame(results)
df.pivot_table(index='status', columns='validation_type', values='message', aggfunc='count').fillna(0.0)

validation_type,comprehensive,sources,target,uniqueness
status,,,,
FAILED,0.0,3.0,1.0,18.0
SUCCESS,105.0,0.0,0.0,0.0


In [17]:
df_success = df[df['status'] == 'SUCCESS'].join(pd.DataFrame(
    df[df['status'] == 'SUCCESS'].pop('metrics').tolist(),
    index=df[df['status'] == 'SUCCESS'].index,
    columns=['precision', 'recall', 'f1_score']
))

df_success['f1_score'].mean()

np.float64(0.9552686912085407)

In [18]:
extractor_pro = SQLLineageExtractor(
    credentials=GIGACHAT_CREDENTIALS,   # or set env var GIGACHAT_API_KEY
    model="GigaChat-Pro",
    verify_ssl_certs=False,
    temperature=0.01
)

In [19]:
data_lineage_pro = []
f1_scores_pro = {}
results_pro = []

for index, row in tqdm(df_data.iterrows(), total=len(df_data), desc="🎨 Extracting S2T"):

    target_ = re_extractor.extract(row['ddl'])

    res_ = await validation.run_comprehensive_validation(
        extractor_pro, 
        row['ddl'],
        expected_result=target_)
    results_pro.append(res_)
    try:
        f1_scores_pro[row['table_name']] = res_['metrics']['f1_score']
    except:
        f1_scores_pro[row['table_name']] = 0
        
    data_lineage_pro.append(res_['result'])

🎨 Extracting S2T:   0%|          | 0/127 [00:00<?, ?it/s]

In [20]:
np.mean(list(f1_scores_pro.values()))

np.float64(0.9217708337828961)

In [21]:
df_pro = pd.DataFrame(results_pro)
df_pro.pivot_table(index='status', columns='validation_type', values='message', aggfunc='count').fillna(0.0)

validation_type,comprehensive,sources,target,uniqueness
status,,,,
FAILED,0.0,2.0,2.0,3.0
SUCCESS,120.0,0.0,0.0,0.0


In [22]:
df_success_pro = df_pro[df_pro['status'] == 'SUCCESS'].join(pd.DataFrame(
    df_pro[df_pro['status'] == 'SUCCESS'].pop('metrics').tolist(),
    index=df_pro[df_pro['status'] == 'SUCCESS'].index,
    columns=['precision', 'recall', 'f1_score']
))

df_success_pro['f1_score'].mean()

np.float64(0.9755407990868984)

In [38]:
pd.DataFrame(df_pro[df_pro.status == 'FAILED'].join(df_data)[['table_name','ddl']]).to_csv(wrong_sql_path, index=False)